In [ ]:
# Montar Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ruta al archivo Excel

file_path = 'data/통합용/final_2023_updated_file.xlsx'

In [ ]:
# Leer el archivo

import pandas as pd

# Leer archivo Excel

df = pd.read_excel(file_path)

# Verificar datos

print("file:")
print(df.head())

In [ ]:
# Definir columnas a mantener
columns_to_keep = ['project_id', 'applicant_id', 'economic_activity', 'question_text', 'answer_text']

# Mantener solo las columnas definidas
df = df[columns_to_keep]

# Eliminar duplicados
df = df.drop_duplicates()

# Eliminar filas sin datos en 'economic_activity'
df = df.dropna(subset=['economic_activity'])

# Verificar datos procesados
df.head()

In [ ]:
# Preparar los datos
# Crear texto combinado con pregunta y respuesta
df['combined_text'] = (
    "Pregunta: " + df['question_text'] +
    "\nRespuesta: " + df['answer_text']
)

# Verificar texto combinado
df[['project_id', 'applicant_id', 'economic_activity', 'combined_text']].head()

In [ ]:
# Verificar valores únicos en 'economic_activity'
print(df.isnull().sum())  # Verificar valores nulos
data = df.dropna()

1.2 Preparación de datos
Asignación de preguntas según criterios

Se clasifican las 9 preguntas en 4 criterios principales.

Ejemplos:

Originalidad del producto/servicio ofrecido: Pregunta1, Pregunta2

Diferenciación del público objetivo: Pregunta3, Pregunta4, Pregunta5

Originalidad en el método de operación: Pregunta6, Pregunta7, Pregunta8

Contribución a la comunidad/localidad o al medio ambiente: Pregunta9

In [ ]:
# Mapeo de criterios a preguntas específicas

criteria_mapping = {
    "Operating model originality": [
        "¿Cuáles son las alianzas realizadas o a realizar para mejorar la satisfacción de mis clientes?",
        "¿A través de qué medios realizo las ventas a mis clientes? ¿Cuáles son los medios, para dar a conocer mi producto/servicio, que prefieren mi/s tipo/s de clientes? ¿Cuáles son los medios con los que obtendría mayor venta en mi modelo de negocio?",
        "¿Qué acciones puedo implementar en mi negocio, desde el punto de vista de la eficiencia energética, energías renovables y economía circular? de manera de hacer mi producto o servicio más sustentable. ¿Tenía ya incorporada alguna de esta acciones en el proceso de mi producto o servicio?"
    ],
    "Customer targeting differentiation": [
        "¿Quiénes son los principales clientes? ¿A qué tipo de clientes apunta nuestro negocio?",
        "¿Qué relación tiene o espera tener con cada tipo de cliente descrito? ¿Alguno de los medios por los cuales busca relacionarse con el cliente, tiene algún costo asociado?",
        "¿Por cuál tipo de producto/servicio estarían dispuestos a pagar más nuestros clientes? ¿Por cuál tipo de producto/servicio pagan actualmente los clientes? ¿Qué tipo de medio de pago prefieren utilizar mis clientes?"
    ],
    "Product/service originality": [
        "¿Por qué los clientes deberían preferirme por sobre los demás?  ¿Por qué los clientes deberían preferir mi producto/servicio por sobre los demás?"
    ]
}

In [ ]:
# Mapeo de criterios a preguntas específicas

criteria_mapping = {
    #Originalidad en el método de operación
    "Operating model originality": [
        "¿Cuáles son las alianzas gestionadas para mejorar la satisfacción de mis clientes a través de la oferta de valor?",
        "¿Qué actividades clave se deben desarrollar para que nuestra oferta de valor llegue a los diferentes segmentos de clientes definidos en el modelo de negocios?",
        "¿Qué recursos clave se deben gestionar para que nuestra oferta de valor llegue a los diferentes segmentos de clientes definidos en el modelo de negocios?",
        "¿A través de qué canales quiero llegar a mis clientes? ¿Cuáles son los canales que funcionan mejor de acuerdo a mi segmento de clientes? ¿Cuáles son los canales más rentables de mi modelo de negocio?"
    ],
    #Diferenciar su público objetivo
    "Customer targeting differentiation": [
        "¿Por qué están dispuestos a pagar nuestros diferentes segmentos de clientes? ¿Por qué pagan actualmente nuestros segmentos potenciales de clientes? ¿Por qué métodos prefieren pagar cada segmento de clientes?",
        "¿Qué relación espera tener con cada segmento de clientes descrito? ¿Cuál es el costo de cada una de las formas de relacionarse con cada segmento?",
        "¿Quiénes son los clientes a los cuales les estamos entregando valor? ¿Cuáles son los segmentos más importantes de clientes a los que apunta nuestro negocio?"
    ],
    #Originalidad del producto/servicio prestado
    "Product/service originality": [
        "¿Por qué debería preferirme el segmento de clientes al que apunta mi idea de negocio, y no quedarse con la competencia?"
    ]
}

In [ ]:
# Agregar columna de criterio

def assign_criteria(question):
    for criteria, questions in criteria_mapping.items():
        if question in questions:
            return criteria
    return "other"

# Convertir columnas de texto a tipo string

df['criteria'] = df['question_text'].apply(assign_criteria)

In [ ]:
# Agrupar datos por postulante y proyecto

df['question_text'] = df['question_text'].astype(str)
df['answer_text'] = df['answer_text'].astype(str)

In [ ]:
print(df)

In [ ]:
# Filtrar solo preguntas con criterios definidos

df_filtered = df[df['criteria'].notnull() & (df['criteria'] != "other")]

# Agrupar manteniendo todas las columnas

grouped_df = (
    df_filtered.groupby(['project_id', 'applicant_id', 'economic_activity', 'criteria'], as_index=False)
    .agg({
        'question_text': lambda x: list(x),
        'answer_text': ' '.join
        })
    )

# Verificar resultados

grouped_df.head()

In [ ]:
# Guardar archivo final

download_file_path = "data/Evaluación de diferenciación/final_2023_grouped_categorized_file.xlsx"
grouped_df.to_excel(download_file_path, index=False)

print(f"File saved : {download_file_path}")